## Programmatic methods to sample & compare NS and SRNS points

In [53]:
# Custom modules with their classes inside
import behaviors
import extract_hyperplanes  # Unfinished module
import no_signaling_sets
import numpy as np


## Experiment parameters

In [54]:
delta = 2 # Number of outputs a,b
m = 2     # Number of inputs x,y

## Get a non-SRNS point from database

In [55]:
data = np.load("../data/non_srns/non_srns_points.npy")
example_point = data[np.random.randint(len(data))]

print("Example point as vector:", example_point)
print("\n\n")
print("Example point as behavior:")
behavior = behaviors.RoutedBehavior(delta, m, example_point)
print(behavior)

Example point as vector: [0.14261609 0.32331493 0.02367068 0.23233624 0.59749494 0.4167961
 0.41954079 0.21087523 0.25726494 0.2015882  0.37621035 0.29256689
 0.00262402 0.05830076 0.18057817 0.26422163 0.70451276 0.41064618
 0.252798   0.4203651  0.03559827 0.32946486 0.19041347 0.02284638
 0.03435598 0.07514589 0.48607074 0.06542697 0.22553298 0.18474307
 0.07071778 0.49136155]



Example point as behavior:
Behavior:
Short path (z=S):
[[0.14261609 0.32331493 0.02367068 0.23233624]
 [0.59749494 0.4167961  0.41954079 0.21087523]
 [0.25726494 0.2015882  0.37621035 0.29256689]
 [0.00262402 0.05830076 0.18057817 0.26422163]]
Long path (z=L) :
[[0.70451276 0.41064618 0.252798   0.4203651 ]
 [0.03559827 0.32946486 0.19041347 0.02284638]
 [0.03435598 0.07514589 0.48607074 0.06542697]
 [0.22553298 0.18474307 0.07071778 0.49136155]]
------------


### Elementary tests on behaviors

In [56]:
print(f"Coordinates are positive                      : {behavior.positivity()}")
print(f"Coordinates are normalized                    : {behavior.normalization()}")
print(f"Coordinates verify the no-signaling conditions: {behavior.no_signaling()}")
print()
print("Aggregated tests (checks all previous tests):")
print(f"  Normalization                               : {behavior.is_normalized()}")
print(f"  No-signaling                                : {behavior.is_no_signaling()}")

Coordinates are positive                      : True
Coordinates are normalized                    : True
Coordinates verify the no-signaling conditions: True

Aggregated tests (checks all previous tests):
  Normalization                               : True
  No-signaling                                : True


## Instanciate a set to test for belonging in SRNS

In [57]:
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)
srns_set

### Pipelined belonging test

In [58]:
print(f"Does SRNS set contains the example point: {srns_set.is_in_set(behavior)}")

Does SRNS set contains the example point: False


### Main steps to belonging test

#### [1] Get the equation to test for belonging in matrix form

In [59]:
A,b = srns_set.get_equations(behavior)

with np.printoptions(threshold=np.inf, linewidth=np.inf, precision=2): # type: ignore
    print("A matrix:")
    print(A[:-4])
    print("Last 4 rows of A matrix")
    print(A[-4:])
    print()
    print("b vector:")
    print(b)

A matrix:
[[ 0.11  1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.07  0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.23  0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.02  0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.35  0.    0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0. 

#### [2] Solve the linear program

In [60]:
from scipy.optimize import OptimizeResult

result: OptimizeResult = srns_set.lp_test(behavior)

In [61]:
print(f"Alpha value for  alpha*p + (1-alpha)*I  : {-result.fun}", "< 1" if -result.fun < 1 else ">= 1")  # noqa: E501

print()

latent_q_vector = result.x[1:]
latent_behavior = behaviors.LatentSRNSBehavior(delta, m, latent_q_vector)
print("Latent behavior:")
print(latent_behavior)
print("Latent behavior is no-signaling: ", latent_behavior.no_signaling())

Alpha value for  alpha*p + (1-alpha)*I  : 0.8979515237673755 < 1

Latent behavior:
Behavior:
Short path (z=S):
[[0.15357446 0.31583325 0.04676725 0.2341388 ]
 [0.56203361 0.39977482 0.40223941 0.21486785]
 [0.25652357 0.20652855 0.36333078 0.288223  ]
 [0.02786837 0.07786338 0.18766256 0.26277034]]
Long path (z=L) :
[[0.33677484 0.25251247]
 [0.32135559 0.        ]
 [0.05747764 0.15046713]
 [0.         0.04602706]
 [0.         0.08426237]
 [0.05636212 0.37771771]
 [0.09298949 0.        ]
 [0.13504032 0.08901326]]
------------
Latent behavior is no-signaling:  True


In [66]:
def format_lambda_to_hyperplane(lam: np.ndarray) -> str:
    return str(lam[:16]) + str(lam[16:32]) + str(lam[32:]) 

In [67]:
_, _, lambda_var = srns_set.is_facet_hyperplane(behavior)
hyperplanes_extractor = extract_hyperplanes.HyperplanesExtractor(delta, m, list(data))
print("Corresponding lambda variable:")
with np.printoptions(threshold=np.inf, precision=2):  # type: ignore
    print(lambda_var)
print(len(lambda_var), "is the number of coordinates in the dual variable")
print()

hyperplane = hyperplanes_extractor.scale_down_vector(lambda_var)
print("Rescaled and sliced to the hyperplane size:")
print(format_lambda_to_hyperplane(hyperplane))

Corresponding lambda variable:
[ 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.   0.   1.8  0.  -1.8  1.8  1.8  0.   0.   0.
  1.8 -1.8  0.   0.   0.   0.  -1.8  0. ]
36 is the number of coordinates in the dual variable

Rescaled and sliced to the hyperplane size:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0][ 0  0  0  0  1  0 -1  1  1  0  0  0  1 -1  0  0][ 0  0 -1  0]
